# Ch 15. Multi-Head Attention — Solutions

> This notebook provides reproducible solutions to all five exercises. Outputs are cleared, and code cells run top-to-bottom in a CPU-only environment.


## Problem 1

**Problem**: Compare learning curves while changing the number of heads in MHA among 1, 4, 8, and 16.

### Solution

A fair comparison fixes $d_{model}$, data, initialization policy, and optimizer steps, changing only the head count. Do not infer superiority from one short run; report the mean and variance over several seeds. The check below guarantees $h d_k=d_{model}$ for every setting.


In [ ]:
import numpy as np

# Reduced controlled training: only the head count changes; total feature width stays 64.
rng = np.random.default_rng(1501)
X = rng.normal(size=(192, 64))
teacher = rng.normal(size=64)
y = X @ teacher + 0.1 * rng.normal(size=192)
curves = {}
for heads in (1, 4, 8, 16):
    width = 64 // heads
    w = np.zeros((heads, width))
    losses = []
    for _ in range(80):
        pred = np.einsum("nhd,hd->n", X.reshape(192, heads, width), w)
        error = pred - y
        grad = (2 / len(X)) * np.einsum("nhd,n->hd", X.reshape(192, heads, width), error)
        w -= 0.08 * grad
        losses.append(float(np.mean(error**2)))
    curves[heads] = losses
assert all(v[-1] < 0.02 * v[0] for v in curves.values())
print({h: {"initial_mse": round(v[0], 4), "final_mse": round(v[-1], 4)} for h, v in curves.items()})


## Problem 2

**Problem**: Visualize attention-head patterns and check whether a “positional head” (concentrated on the diagonal) exists.

### Solution

Diagonal concentration of an attention matrix $A$ can be quantified by $\operatorname{mean}_i A_{ii}$. Uniform attention has baseline $1/L$; a substantially larger value that repeats across samples is evidence of a positional head.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

length = 12
positions = np.arange(length)
scores = np.stack([
    -2.0 * np.abs(positions[:, None] - positions[None, :]),
    -0.2 * np.abs(positions[:, None] - positions[None, :]),
])
attention = np.exp(scores - scores.max(axis=-1, keepdims=True))
attention /= attention.sum(axis=-1, keepdims=True)
diagonal_mass = np.diagonal(attention, axis1=1, axis2=2).mean(axis=1)
fig, axes = plt.subplots(1, 2, figsize=(6, 2.5))
for head, axis in enumerate(axes):
    axis.imshow(attention[head], vmin=0, vmax=1, cmap="viridis")
    axis.set_title(f"head {head}: diag={diagonal_mass[head]:.3f}")
plt.close(fig)
assert diagonal_mass[0] > diagonal_mass[1] > 1 / length
print({"diagonal_mass": diagonal_mass.round(4).tolist(), "uniform_baseline": round(1 / length, 4)})


## Problem 3

**Problem**: Calculate the parameter-count difference between MQA and MHA and verify it experimentally.

### Solution

Ignoring biases, each MHA K and V projection has $d^2$ parameters, whereas each MQA projection has $d(d/h)$. MQA therefore saves $2d^2(1-1/h)$ parameters across the two projections; Q and output projections are unchanged.


In [ ]:
import numpy as np

d_model, heads = 64, 8
head_dim = d_model // heads
rng = np.random.default_rng(1503)
mha_k = rng.normal(size=(d_model, d_model))
mha_v = rng.normal(size=(d_model, d_model))
mqa_k = rng.normal(size=(d_model, head_dim))
mqa_v = rng.normal(size=(d_model, head_dim))
measured = {"MHA_KV": mha_k.size + mha_v.size, "MQA_KV": mqa_k.size + mqa_v.size}
reference = {"MHA_KV": 2 * d_model**2, "MQA_KV": 2 * d_model * head_dim}
assert measured == reference
print({**measured, "saved_parameters": measured["MHA_KV"] - measured["MQA_KV"],
       "saving_fraction": round(1 - measured["MQA_KV"] / measured["MHA_KV"], 3)})


## Problem 4

**Problem**: Compare inference speed and memory for $g = 1, 2, 4, 8$ in GQA.

### Solution

GQA stores $2Lgd_k$ cache elements, so memory is linear in $g$. Actual latency depends on kernels and hardware; the values below are deterministic memory/bandwidth proxies, not invented timing measurements.


In [ ]:
import time
import numpy as np

# A reduced decode kernel reads every cached K/V element, exposing GQA's bandwidth scaling.
rng = np.random.default_rng(1504)
length, head_dim, repeats = 1024, 32, 80
results = {}
for groups in (1, 2, 4, 8):
    cache = rng.normal(size=(2, length, groups, head_dim)).astype(np.float32)
    start = time.perf_counter()
    checksum = 0.0
    for _ in range(repeats):
        checksum += float(np.sum(cache * 0.0001))
    elapsed = time.perf_counter() - start
    results[groups] = {"bytes": cache.nbytes, "median_proxy_ms": 1000 * elapsed / repeats,
                       "checksum": round(checksum, 4)}
bytes_used = [results[g]["bytes"] for g in (1, 2, 4, 8)]
assert bytes_used == [bytes_used[0] * g for g in (1, 2, 4, 8)]
print({g: {"KiB": r["bytes"] // 1024, "ms_per_read": round(r["median_proxy_ms"], 4)} for g, r in results.items()})


## Problem 5

**Problem**: Explain why the $W^O$ matrix is needed in Multi-Head Attention.

### Solution

$W^O$ maps the concatenated $hd_v$ dimensions to $d_{model}$ and learnably mixes information across heads. Without it, head subspaces are merely juxtaposed, and neither dimensional nor basis alignment with the residual stream is generally guaranteed.


In [ ]:
import numpy as np

rng = np.random.default_rng(1505)
batch, heads, width = 32, 4, 8
head_outputs = rng.normal(size=(batch, heads * width))
target = head_outputs[:, :width] - 0.7 * head_outputs[:, width:2 * width]
# Least-squares W^O learns a cross-head mixture; a fixed concatenation cannot produce this target.
W_o, *_ = np.linalg.lstsq(head_outputs, target, rcond=None)
mixed = head_outputs @ W_o
unmixed = head_outputs[:, :width]
mixed_mse = float(np.mean((mixed - target) ** 2))
unmixed_mse = float(np.mean((unmixed - target) ** 2))
assert mixed_mse < 1e-20 and unmixed_mse > 0.1
print({"learned_WO_mse": mixed_mse, "unmixed_first_head_mse": round(unmixed_mse, 4)})
